In [1]:
# Cell 1 — Imports & DB connection
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"C:\Users\Negza\Desktop\projects\pfe\bvmt_project\.env")

conn = psycopg2.connect(
    host=os.getenv('DB_HOST'), port=os.getenv('DB_PORT'),
    dbname=os.getenv('DB_NAME'), user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)
print("Connected to", os.getenv('DB_NAME'))


Connected to bvmt_db


In [2]:
# Cell 2 — RSI (pure pandas, Wilder EMA — same algorithm pandas-ta used)
def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


In [3]:
# Cell 3 — Load prices
df_all = pd.read_sql('''
    SELECT seance, isin_code, ticker, cloture, quantite_negociee
    FROM daily_prices
    WHERE cloture > 0
    ORDER BY isin_code, seance
''', conn)
df_all['seance'] = pd.to_datetime(df_all['seance'])
print(f"Loaded {len(df_all):,} rows for {df_all['isin_code'].nunique()} stocks")

C:\Users\Negza\AppData\Local\Temp\ipykernel_5736\3442129687.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql('''


Loaded 116,629 rows for 121 stocks


In [4]:
# Cell 4 — Compute indicators per stock
def compute_indicators_for_stock(df_stock):
    df = df_stock.copy().set_index('seance').sort_index()
    close  = df['cloture']
    volume = df['quantite_negociee']

    df['ma_5']          = close.rolling(5).mean()
    df['ma_20']         = close.rolling(20).mean()
    df['ma_50']         = close.rolling(50).mean()
    df['rsi_14']        = compute_rsi(close, period=14)
    df['daily_return']  = close.pct_change()
    df['volatility_20'] = df['daily_return'].rolling(20).std()
    df['volume_ma_20']  = volume.rolling(20).mean()
    df['price_to_ma20'] = close / df['ma_20']

    return df.reset_index()

In [5]:
# Cell 5 — Process each stock and insert into DB
all_rows = []

for isin, group in df_all.groupby('isin_code'):
    df_ind = compute_indicators_for_stock(group)
    ticker = group['ticker'].iloc[0]

    for row in df_ind.itertuples():
        if pd.notna(row.ma_20):
            all_rows.append((
                row.seance.date(), isin, ticker,
                row.ma_5, row.ma_20, row.ma_50,
                row.rsi_14        if pd.notna(row.rsi_14)        else None,
                row.daily_return  if pd.notna(row.daily_return)  else None,
                row.volatility_20 if pd.notna(row.volatility_20) else None,
                row.volume_ma_20  if pd.notna(row.volume_ma_20)  else None,
                row.price_to_ma20 if pd.notna(row.price_to_ma20) else None,
            ))

print(f"Ready to insert {len(all_rows):,} indicator rows")
#bulk insert 

sql = '''
    INSERT INTO computed_indicators
        (seance, isin_code, ticker, ma_5, ma_20, ma_50,
         rsi_14, daily_return, volatility_20, volume_ma_20, price_to_ma20)
    VALUES %s
    ON CONFLICT (seance, isin_code) DO UPDATE SET
        ma_5 = EXCLUDED.ma_5, ma_20 = EXCLUDED.ma_20, ma_50 = EXCLUDED.ma_50,
        rsi_14 = EXCLUDED.rsi_14, daily_return = EXCLUDED.daily_return,
        volatility_20 = EXCLUDED.volatility_20, volume_ma_20 = EXCLUDED.volume_ma_20,
        price_to_ma20 = EXCLUDED.price_to_ma20
'''

with conn.cursor() as cur:
    execute_values(cur, sql, all_rows, page_size=2000)
conn.commit()
print(f"Done! {len(all_rows):,} indicator rows inserted/updated.")

Ready to insert 114,346 indicator rows
Done! 114,346 indicator rows inserted/updated.


In [6]:
# Cell 6 — Recompute only for BH BANK after price reload
# This cell reuses existing notebook objects: conn, compute_indicators_for_stock, execute_values.

TICKER_FILTER = 'BH BANK'
BH_ISIN = 'TN0001900604'

# 1) Remove old indicators for BH BANK so the recompute starts clean.
with conn.cursor() as cur:
    cur.execute("DELETE FROM computed_indicators WHERE ticker = %s", (TICKER_FILTER,))
conn.commit()
print(f"Deleted old indicators for {TICKER_FILTER}")

# 2) Load only BH BANK prices from daily_prices.
df_bh = pd.read_sql(
    '''
    SELECT seance, isin_code, ticker, cloture, quantite_negociee
    FROM daily_prices
    WHERE ticker = %s AND isin_code = %s AND cloture > 0
    ORDER BY seance
    ''',
    conn,
    params=(TICKER_FILTER, BH_ISIN),
)
df_bh['seance'] = pd.to_datetime(df_bh['seance'])
print(f"Loaded {len(df_bh):,} BH BANK price rows")

# 3) Compute indicators only for BH BANK and prepare insert payload.
all_rows = []
if not df_bh.empty:
    df_ind = compute_indicators_for_stock(df_bh)

    for row in df_ind.itertuples():
        if pd.notna(row.ma_20):
            all_rows.append((
                row.seance.date(), BH_ISIN, TICKER_FILTER,
                row.ma_5, row.ma_20, row.ma_50,
                row.rsi_14        if pd.notna(row.rsi_14)        else None,
                row.daily_return  if pd.notna(row.daily_return)  else None,
                row.volatility_20 if pd.notna(row.volatility_20) else None,
                row.volume_ma_20  if pd.notna(row.volume_ma_20)  else None,
                row.price_to_ma20 if pd.notna(row.price_to_ma20) else None,
            ))

print(f"Ready to insert {len(all_rows):,} BH BANK indicator rows")

# 4) Upsert BH BANK indicators (duplicate-safe via ON CONFLICT).
if all_rows:
    sql = '''
        INSERT INTO computed_indicators
            (seance, isin_code, ticker, ma_5, ma_20, ma_50,
             rsi_14, daily_return, volatility_20, volume_ma_20, price_to_ma20)
        VALUES %s
        ON CONFLICT (seance, isin_code) DO UPDATE SET
            ma_5 = EXCLUDED.ma_5, ma_20 = EXCLUDED.ma_20, ma_50 = EXCLUDED.ma_50,
            rsi_14 = EXCLUDED.rsi_14, daily_return = EXCLUDED.daily_return,
            volatility_20 = EXCLUDED.volatility_20, volume_ma_20 = EXCLUDED.volume_ma_20,
            price_to_ma20 = EXCLUDED.price_to_ma20
    '''

    with conn.cursor() as cur:
        execute_values(cur, sql, all_rows, page_size=2000)
    conn.commit()

print(f"Done! {len(all_rows):,} BH BANK indicator rows inserted/updated.")

Deleted old indicators for BH BANK
Loaded 2,488 BH BANK price rows
Ready to insert 2,469 BH BANK indicator rows


C:\Users\Negza\AppData\Local\Temp\ipykernel_5736\2511488718.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bh = pd.read_sql(


Done! 2,469 BH BANK indicator rows inserted/updated.
